# **🔍 Clean & Jerk Dataset — Per-Annotator SD Attribution**

Compares each individual annotator's contributed inter-annotator SD distribution against the **Overall pooled SD distribution** to identify whether specific annotators drive higher disagreement.

**Analysis Pipeline:**
`sets/set_XXXX.h5 (/phase_markers/consensus + /annotators)` → `load_annotator_marker_records` →
1. **Attribution Table:** Per-annotator contributed SD stats + Leave-One-Annotator-Out (LOAO) $\Delta\text{SD}$ + Peer Bias.
2. **Small Multiples Histogram:** Each annotator's normalized contributed SD histogram overlaid on the Overall pooled distribution.
3. **Boxplots + Overlaid ECDFs:** Direct side-by-side distribution & cumulative agreement curves vs. Overall.
4. **Causal LOAO Attribution:** $\Delta\text{SD}_{i,a} = \text{SD}_i - \text{SD}_{i,-a}$ and signed peer deviation $t_{i,a} - \bar{t}_{i,-a}$ (separates annotators who *cause* SD inflation from those who simply annotated harder sets).

### 🔧 Colab Setup & Library Init

In [ ]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import (
    DEFAULT_CLEAN_DATASET_DIR,
    build_annotator_sd_attribution_table,
    load_annotator_marker_records,
    plot_annotator_loao_impact,
    plot_annotator_sd_ecdf_and_box,
    plot_annotator_sd_small_multiples,
)

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
EXPORT_CFG = {"toImageButtonOptions": {"format": "png", "scale": 4}}

### 📥 Load Per-Annotator Phase Marker Records (All Sets)

In [ ]:
# @title 📥 Load Marker Records Once (Fast — Reads Phase Markers Only)
min_annotators = 3 # @param {type:"integer"}
records = load_annotator_marker_records(dataset_dir, min_annotators=min_annotators)
print(
    f"Loaded {len(records['consensus_df']):,} consensus marker instances "
    f"and {len(records['contrib_df']):,} annotator contributions across "
    f"{records['contrib_df']['annotator_id'].nunique()} annotators."
)

### 📋 Per-Annotator SD Attribution Table

- **Median SD [IQR] / 95th pct:** Distribution of consensus `sd_ms` on instances this annotator contributed to.
- **Median SD w/o Ann:** Leave-One-Annotator-Out SD ($\text{SD}_{-a}$) after removing this annotator.
- **Median / Mean $\Delta\text{SD}$:** $\text{SD}_{\text{all}} - \text{SD}_{-a}$ — positive values indicate this annotator **increased** SD.
- **Mean Peer Bias:** Mean of $(t_{a} - \bar{t}_{-a})$ — detects systematic early (`-`) or late (`+`) marking bias.

In [ ]:
# @title 📋 Per-Annotator SD Attribution Summary Table
threshold_ms = 200.0 # @param {type:"number"}
attr_df = build_annotator_sd_attribution_table(records, min_annotators=min_annotators, threshold_ms=threshold_ms)
display_table(attr_df, clean_columns=False)

### 📊 Plot 1: Individual Annotator SD Distributions vs. Overall Pooled Distribution

In [ ]:
# @title 📊 Small Multiples: Each Annotator's Contributed SD vs. Overall
bin_width_ms = 20.0 # @param {type:"number"}
cols = 3 # @param {type:"integer"}
x_max_ms = 0 # @param {type:"number"}  # 0 = auto (99.5th pct)

fig_sm = plot_annotator_sd_small_multiples(
    records,
    bin_width_ms=bin_width_ms,
    min_annotators=min_annotators,
    x_max_ms=None if x_max_ms <= 0 else x_max_ms,
    cols=cols,
)
fig_sm.show(config=EXPORT_CFG)

### 📈 Plot 2: Direct Comparison — Boxplots & Overlaid ECDFs vs. Overall

In [ ]:
# @title 📈 Per-Annotator Boxplots & Overlaid ECDFs vs. Overall
x_max_ecdf_ms = 0 # @param {type:"number"}  # 0 = auto (99th pct)

fig_ecdf = plot_annotator_sd_ecdf_and_box(
    records,
    min_annotators=min_annotators,
    x_max_ms=None if x_max_ecdf_ms <= 0 else x_max_ecdf_ms,
)
fig_ecdf.show(config=EXPORT_CFG)

### 🎯 Plot 3: Causal Leave-One-Annotator-Out ($\Delta\text{SD}$) & Signed Peer Bias

In [ ]:
# @title 🎯 Causal LOAO SD Impact & Signed Peer Deviation
fig_loao = plot_annotator_loao_impact(records, min_annotators=min_annotators)
fig_loao.show(config=EXPORT_CFG)